In [1]:
import pandas as pd 
import gffutils

In [6]:
BASE_DIR = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION"
ATSE_FILE_PATH = (
    f"{BASE_DIR}/ATSE_mapper/ATSE_files/TMS_atse_file_unanno_also_2025-07-31_08-10-26.txt.gz"
)

# Load ATSE file (gzip inferred from .gz)
atse_df = pd.read_csv(ATSE_FILE_PATH, sep="\t")

# Sanity check expected columns
expected = {"event_id", "junction_id", "gene_id", "perfect_match_5_prime", "perfect_match_3_prime"}
missing = expected - set(atse_df.columns)
if missing:
    raise KeyError(f"Missing columns in ATSE file: {sorted(missing)}")

# Annotate junctions
atse_df["junction_annotation"] = "Novel_SS"
atse_df.loc[atse_df["perfect_match_5_prime"].notna(), "junction_annotation"] = "5_prime_annotated"
atse_df.loc[atse_df["perfect_match_3_prime"].notna(), "junction_annotation"] = "3_prime_annotated"
atse_df.loc[
    atse_df["perfect_match_5_prime"].notna() & atse_df["perfect_match_3_prime"].notna(),
    "junction_annotation"
] = "Both_SS_annotated"

# Prints (fixed quoting)
print(f'Number of unique ATSEs here is {atse_df["event_id"].nunique()}')
print(f'Number of unique junctions here is {atse_df["junction_id"].nunique()}')
print(f'Number of unique genes here is {atse_df["gene_id"].nunique()}')


Number of unique ATSEs here is 41489
Number of unique junctions here is 160252
Number of unique genes here is 13668


In [10]:
# Load in GTEx junctions 
gtex_junctions_coordinates = pd.read_csv(
    "/gpfs/commons/groups/knowles_lab/Karin/data/GTEx/v10/gtex_junctions_coordinates.tsv",
    sep="\t", header=None
)

In [44]:
# Clean up the GTEx junctions coordinates

pat = r'^(chr[0-9A-Za-z._]+)[:_](\d+)[-_](\d+)[:_]?([+-])$'

out = gtex_junctions_coordinates.copy()
out[["chrom","start","end","strand"]] = out[0].str.extract(pat)
out["start"] = out["start"].astype(int) - 1
out["end"]   = out["end"].astype(int) 

# Build GTEx/ATSE-style junction_id: chr_start_end_strand (underscores)
out["junction_id"] = (
    out["chrom"] + "_" + out["start"].astype(str) + "_" + out["end"].astype(str) + "_" + out["strand"]
)

# Gene id from column 1
out["gene_id"] = out[1]
out["original_junction_id"] = out[0]

# Keep/rename columns to mirror atse_df schema
cols = ["junction_id", "chrom", "start", "end", "strand", "gene_id", "original_junction_id"]
out = out[cols].rename(columns={"strand": "strand"})  # keep name consistent

In [46]:
# check overlap as is with coordinates 
print(out[out["junction_id"].isin(atse_df["junction_id"])].shape[0]/out.shape[0])
print(atse_df[atse_df["junction_id"].isin(out["junction_id"])].shape[0]/atse_df.shape[0])

0.2702752223537046
0.662743678706038


In [48]:
# save just the junctions that overlap with ATSEs
out[out["junction_id"].isin(atse_df["junction_id"])].to_csv(
    "/gpfs/commons/groups/knowles_lab/Karin/data/GTEx/v10/gtex_junctions_coordinates_atse_overlap.tsv",
    sep="\t", index=False
)